GEETA 4B — C4 · Inference client (llama.cpp /v1, OpenAI-compatible).

Talks to a local llama-server (config: model.server_base_url). The SAME
interface is used for:
* real GEETA 4B GGUF weights (M4 output) — text + vision (mtmd)
* the deterministic MockModel (config allow_mock_model) — a label-carrying
  tutor simulator for building/testing M5–M9 while no local weights exist.

Vision: images are sent as base64 data URLs in the OpenAI chat format —
llama.cpp mtmd consumes exactly that.

## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import base64
import json
import time
from pathlib import Path
from typing import Any, Optional

import requests

from geeta.tutor_data import TUTOR_SYSTEM

In [ ]:


def load_config(root: str | Path = ".") -> dict:
    import yaml
    p = Path(root) / "config" / "config.yaml"
    with open(p) as f:
        return yaml.safe_load(f)

In [ ]:


class ModelClient:
    """OpenAI-compatible client for llama-server (or any /v1 backend)."""

    def __init__(self, base_url: str, *, api_key: str = "geeta-local",
                 temperature: float = 0.6, top_p: float = 0.9, top_k: int = 40,
                 max_tokens: int = 512, timeout_s: float = 120.0,
                 repeat_penalty: float = 1.1, enable_thinking: bool = False):
        self.base_url = base_url.rstrip("/")
        self.api_key = api_key
        self.temperature = temperature
        self.top_p = top_p
        self.top_k = top_k
        self.max_tokens = max_tokens
        self.timeout_s = timeout_s
        self.repeat_penalty = repeat_penalty
        # Training used tokenizer.apply_chat_template(..., enable_thinking=False)
        # for every row (see notebooks/04_finetune.ipynb Cell F). The Qwen3.5
        # chat template branches on this flag: leaving it unset defaults to
        # thinking ON, which is off-distribution for our LoRA and burns the
        # whole max_tokens budget on <think>...</think> reasoning, leaving
        # message.content empty. Must match training: False.
        self.enable_thinking = enable_thinking
        self.last_metrics = {"ttft_ms": None, "gen_ms": None, "tokens": None,
                             "tokens_per_s": None}

    # -- low level ---------------------------------------------------------
    def _chat(self, messages: list[dict], **over) -> dict:
        body = {
            "model": over.get("model", "geeta-4b"),
            "messages": messages,
            "temperature": over.get("temperature", self.temperature),
            "top_p": over.get("top_p", self.top_p),
            "top_k": over.get("top_k", self.top_k),
            "max_tokens": over.get("max_tokens", self.max_tokens),
            "repeat_penalty": over.get("repeat_penalty", self.repeat_penalty),
            "chat_template_kwargs": {
                "enable_thinking": over.get("enable_thinking", self.enable_thinking)},
            "stream": False,
        }
        t0 = time.perf_counter()
        r = requests.post(self.base_url + "/v1/chat/completions",
                          headers={"Authorization": f"Bearer {self.api_key}"},
                          json=body, timeout=self.timeout_s)
        r.raise_for_status()
        gen_ms = (time.perf_counter() - t0) * 1000
        data = r.json()
        n = (data.get("usage") or {}).get("completion_tokens")
        self.last_metrics = {"ttft_ms": round(gen_ms, 1), "gen_ms": round(gen_ms, 1),
                             "tokens": n,
                             "tokens_per_s": round(n / (gen_ms / 1000), 1)
                             if n and gen_ms else None}
        return data

    def is_alive(self) -> bool:
        try:
            return requests.get(self.base_url + "/health", timeout=3).ok
        except requests.RequestException:
            return False

    # -- high level --------------------------------------------------------
    def chat(self, messages: list[dict], **over) -> str:
        data = self._chat(messages, **over)
        msg = data["choices"][0]["message"]
        content = (msg.get("content") or "").strip()
        if content:
            return content
        # Defensive fallback only: with enable_thinking=False (the default)
        # content should never be empty. If a caller overrides thinking back
        # on, or a future template changes defaults, surface the reasoning
        # trace rather than silently returning "" to the tutor session.
        return (msg.get("reasoning_content") or "").strip()

    def chat_image(self, image_path: str | Path, prompt: str, **over) -> str:
        """OCR-style call: image + prompt (mtmd vision path)."""
        p = Path(image_path)
        b64 = base64.b64encode(p.read_bytes()).decode()
        msg = {"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            {"type": "text", "text": prompt},
        ]}
        # R1 (session 12): chat() takes `messages: list[dict]` - this used to
        # pass the single message dict unwrapped, so the request body's
        # "messages" field was a bare object instead of an array, and
        # llama-server 400'd every real ocr_page()/chat_image() call (never
        # caught before because nothing exercised this path against a real
        # server until M6's first real run).
        return self.chat([msg], **over)

    def ocr_page(self, image_path: str | Path) -> str:
        # faithful transcription: near-greedy, no repeat penalty (real pages
        # legitimately repeat numbers/words), full token budget for long pages.
        return self.chat_image(image_path,
            "Transcribe this maths homework exactly as written. Output only the "
            "transcription, one question per line, no commentary.",
            temperature=0.1, top_p=0.85, top_k=20, repeat_penalty=1.0, max_tokens=512)


In [ ]:


# ---------------------------------------------------------------------------
# Deterministic mock tutor (allow_mock_model) — CLEARLY LABELLED in logs
# ---------------------------------------------------------------------------

class MockModel:
    """Deterministic stand-in for GEETA 4B for offline pipeline testing.

    It is a SIMULATOR of the trained behaviour contract, not the model:
    * asks the child for their approach first
    * walks the problem's hint ladder (bank metadata)
    * never reveals the final answer (refusal patterns)
    * emits verifier-checkable JSON steps for its "background working"
      (fault_injection can corrupt a step to exercise the retry loop)

    Every call is labelled [MOCK-MODEL] in the transcript log. It exists so
    M5–M9 can be built and tested end-to-end without local weights; any
    number produced by it must be treated as a mock result, never as a
    measured model performance figure.
    """

    def __init__(self, problem: dict | None = None, *, fault_injection: bool = False,
                 reprompt_message: str = ""):
        self.problem = problem or {}
        self.fault_injection = fault_injection
        self.hint_idx = 0
        self.kid_has_answer = False
        self.reprompt_message = reprompt_message or (
            "I love that you're keen to find out! But the fun is working it out "
            "yourself. Let's try the first little step together.")
        self.calls = 0

    def reset(self, problem: dict | None = None) -> None:
        if problem is not None:
            self.problem = problem
        self.hint_idx = 0
        self.kid_has_answer = False

    # -- behaviour ---------------------------------------------------------
    def respond(self, user_text: str, child_visible: str = "") -> str:
        """Replies to the child's latest message (plain text, no images here)."""
        self.calls += 1
        t = user_text.strip()
        low = t.lower()
        from geeta.leak_guard import answer_in_text, parse_answer_value
        ans = str(self.problem.get("answer") or "").strip()

        # child stated the answer?
        if ans and ans not in ("…", "?", "None") and self._child_may_know(t, ans):
            self.kid_has_answer = True

        # probing for the answer (adversarial / begging)?
        if self._is_probe(low):
            return self.reprompt_message

        if self.kid_has_answer and self._looks_like_final(t):
            return ("YES! That's exactly it — and YOU got there. "
                    "Tell me: what was the trickiest little part? "
                    "I'm putting that in your report. 🌟")

        if self._looks_like_final(t):
            # child stated SOME number but not the answer: coach a re-check,
            # never confirm the wrong value, never state the right one
            return ("Ooh, you've got a number in there! Let's double-check it: "
                    "try it a second way (backwards, or on your fingers) — what do you get?")

        if "stuck" in low or "don't know" in low or "idk" in low or "help" in low:
            hints = self.problem.get("hint_ladder") or []
            if self.hint_idx < len(hints):
                h = hints[self.hint_idx]
                self.hint_idx += 1
                return f"Here's a nudge: {h} Try that and tell me what happens."
            sw = self.problem.get("similar_worked")
            if sw:
                return ("Let's look at a SIMILAR one first: "
                        f"{sw} Now back to YOURS — what's the first little step?")
            return "Let's slow down. Break it into the smallest piece and do that one."

        if self.hint_idx == 0 and not self.problem.get("hint_ladder"):
            return "How would YOU start? Try it on your fingers, or tell me your idea."

        return "Tell me what you tried — even a wrong start is useful!"

    def background_working(self, kid_visible: str) -> str:
        """The model's private step-by-step working (JSON for the verifier).
        Mock derives it from the problem's ground-truth solution_steps."""
        steps = self.problem.get("solution_steps") or []
        if not steps:
            # synthesize a trivial single step from the expression form
            return json.dumps({"steps": [{"step": 1, "expression": "0 + 0",
                                          "result": 0,
                                          "explanation": "no steps available"}],
                               "final_answer": self.problem.get("answer")})
        out = []
        for i, s in enumerate(steps, start=1):
            expr = str(s.get("expression", ""))
            res = s.get("result")
            if self.fault_injection and i == len(steps):
                res = self._corrupt(res)  # deliberate error -> verifier must catch
            out.append({"step": i, "expression": expr, "result": res,
                        "explanation": s.get("explanation", "")})
        return json.dumps({"steps": out,
                           "final_answer": self.problem.get("answer")})

    def ocr(self, image_path: str | Path, transcript_hint: str | None = None) -> str:
        """Mock OCR: the synthetic handwriting PNGs encode the question in the
        file name/manifest; real OCR comes from the mtmd path on GGUF."""
        return transcript_hint or "I can't quite read this — can you tell me what it says?"

    # -- helpers -----------------------------------------------------------
    @staticmethod
    def _corrupt(res: Any) -> Any:
        """Deliberately wrong value (for retry-loop testing)."""
        import re as _re
        if isinstance(res, (int, float)) and not isinstance(res, bool):
            return res + 1
        if isinstance(res, str) and _re.fullmatch(r"\d{1,2}:\d{2}", res.strip()):
            h, m = res.strip().split(":")
            return f"{int(h) + 1}:{m}"
        try:
            return float(str(res).replace(",", "")) + 1
        except (TypeError, ValueError):
            return None

    @staticmethod
    def _is_probe(low: str) -> bool:
        probes = ["just tell me", "tell me the answer", "what's the answer",
                  "the answer is", "give me the answer", "i don't want to work",
                  "stop asking", "just say", "whisper it", "only for me",
                  "promise you won't tell", "cheat", "secret"]
        return any(p in low for p in probes)

    @staticmethod
    def _looks_like_final(t: str) -> bool:
        from geeta.leak_guard import _NUM_TOK
        return bool(_NUM_TOK.search(t)) or "it's" in t.lower() or "i got" in t.lower()

    @staticmethod
    def _child_may_know(t: str, ans: str) -> bool:
        from geeta.leak_guard import answer_in_text, parse_answer_value
        kind, val = parse_answer_value(ans)
        return answer_in_text(kind, val, ans, t)

In [ ]:


def get_model(cfg: dict, problem: dict | None = None,
              fault_injection: bool = False):
    """Factory: real client if a local server is up, else labelled mock."""
    base = cfg["model"]["server_base_url"]
    client = ModelClient(
        base,
        temperature=cfg["serving"]["temperature"],
        top_p=cfg["serving"]["top_p"],
        top_k=cfg["serving"]["top_k"],
        max_tokens=cfg["serving"]["max_tokens"],
        repeat_penalty=cfg["serving"].get("repeat_penalty", 1.1))
    if client.is_alive():
        print(f"[MODEL] using served model at {base} (real weights)")
        return client
    if not cfg["model"].get("allow_mock_model", False):
        raise SystemExit(f"no model server at {base} and allow_mock_model=false")
    print(f"[MODEL] WARNING: no server at {base} — using MOCK tutor simulator "
          f"(results are pipeline tests, NOT model performance)")
    return MockModel(problem, fault_injection=fault_injection)